In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets
import os
import zipfile
from torchvision.datasets import ImageFolder

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
zip_path = "/content/drive/MyDrive/Road_Issues.zip"
extract_path = "/content/extracted_dataset"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [5]:
!pip install -q split-folders

In [6]:
import splitfolders
data_dir = "/content/extracted_dataset/Road Issues"
split_dir = "/content/split_dataset"
splitfolders.ratio(data_dir,output=split_dir,ratio=(0.70, 0.15, 0.15))



train_dir = "/content/split_dataset/train"
val_dir = "/content/split_dataset/val"
test_dir = "/content/split_dataset/test"

Copying files: 6113 files [00:01, 4207.73 files/s]


In [7]:
batch_size = 64
learning_rate = 0.001
num_epochs = 10
num_classes = 5

In [8]:
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),


    transforms.ToTensor(),


    transforms.Normalize(
        mean=(0.4581, 0.4506, 0.4425),
        std=(0.2835, 0.2802, 0.2848)
    )
])

In [9]:
transform_val_test = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

   transforms.Normalize(
        mean=(0.4581, 0.4506, 0.4425),
        std=(0.2835, 0.2802, 0.2848)
    )
])

In [ ]:
train_dataset = ImageFolder(train_dir,transform=train_transform)
val_dataset = ImageFolder(val_dir,transform=transform_val_test)
test_dataset = ImageFolder(test_dir,transform=transform_val_test)

In [11]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)

In [12]:

print("\nTrain images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

print("\nNumber of classes:", len(train_dataset.classes))

print("\nNumber of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))


Train images: 4276
Validation images: 914
Test images: 923

Number of classes: 5

Number of training batches: 134
Number of validation batches: 29
Number of test batches: 29


In [13]:
images, labels = next(iter(train_loader))

print("\nOne batch:")
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)


One batch:
Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])


In [14]:
classes = ('Broken Road Sign Issues', 'Damaged Road issues', 'Illegal Parking Issues', 'Mixed Issues', 'Pothole Issues')

In [15]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=5):

        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),   # 224 -> 112


            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),   # 112 -> 56


            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),   # 56 -> 28
        )

        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 4 * 4, 256),

            nn.ReLU(inplace=True),

            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )


    def forward(self, x):

        x = self.features(x)

        x = self.pool(x)

        x = self.classifier(x)

        return x

In [16]:
import gc

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [17]:

model = SimpleCNN(num_classes=num_classes).to(device)

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [19]:
def train_model():

    # Mixed precision scaler
    scaler = torch.amp.GradScaler("cuda")

    model.train()

    for epoch in range(num_epochs):

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Mixed precision
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(images)
                loss = criterion(outputs, labels)

            # Backpropagation
            scaler.scale(loss).backward()

            scaler.step(optimizer)
            scaler.update()

            # Statistics
            running_loss += loss.item() * images.size(0)

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = 100.0 * correct / total

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Loss: {epoch_loss:.4f} "
            f"Accuracy: {epoch_acc:.2f}%"
        )

In [20]:
def evaluate_model():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(images)

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    acc = 100.0 * correct / total

    print(f"Test Accuracy: {acc:.2f}%")

    return acc




In [21]:
if __name__ == "__main__":
    train_model()
    evaluate_model()

    # Save the trained model
    torch.save(model.state_dict(), "road_model.pth")
    print("Model saved as road_model.pth")

Epoch [1/10] Loss: 0.6841 Accuracy: 79.02%
Epoch [2/10] Loss: 0.5552 Accuracy: 82.37%
Epoch [3/10] Loss: 0.5207 Accuracy: 83.49%
Epoch [4/10] Loss: 0.5025 Accuracy: 84.17%
Epoch [5/10] Loss: 0.4764 Accuracy: 84.99%
Epoch [6/10] Loss: 0.4976 Accuracy: 83.93%
Epoch [7/10] Loss: 0.4611 Accuracy: 85.36%
Epoch [8/10] Loss: 0.4591 Accuracy: 85.55%
Epoch [9/10] Loss: 0.4813 Accuracy: 84.82%
Epoch [10/10] Loss: 0.4537 Accuracy: 85.38%
Test Accuracy: 85.70%
Model saved as road_model.pth
